In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import pandas as pd



# Uncomment the line below to run Chrome in headless mode
# # chrome_options.add_argument("--headless")
driver = webdriver.Chrome()


driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")

wait = WebDriverWait(driver, 20)

advance_btn = wait.until(
EC.element_to_be_clickable(
    (By.XPATH, "//button[contains(., 'Show Advance Search')]")
    )
    )

advance_btn.click()

In [4]:

# -----------------------------
# Select Major Clearance Type
# -----------------------------
major_clearance = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='majorClearanceType']")
    )
)

Select(major_clearance).select_by_value("1")


# -----------------------------
# Search fro Steel
# -----------------------------

text_box = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//input[@formcontrolname='text']")
    )
)

text_box.clear()
text_box.send_keys("Steel")



# -----------------------------
# Select State
# -----------------------------
state_dropdown = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='state']")
    )
)

Select(state_dropdown).select_by_value("28")

# -----------------------------
# Click Search Button
# -----------------------------
search_button = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//button[@type='submit' and contains(.,'Search')]")
    )
)

search_button.click()

print("Search applied successfully.")

# Wait for table to load
wait.until(
    EC.presence_of_element_located(
        (By.ID, "excel-table")
    )
)

Search applied successfully.


<selenium.webdriver.remote.webelement.WebElement (session="67029e276d9f8161a1b073aee38f81a4", element="f.FA5253BB33B88E0C9DE6E09E961957F0.d.5BB2B9A65E636CAB0166ADC6D3A49D72.e.107")>

In [73]:

  
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')
table = soup.find("table", {"id": "excel-table"})
# products = []
  
 # Extract headers
headers = []
for th in table.find("thead").find_all("th"):
    headers.append(th.get_text(strip=True))
  
# print(headers)
# Extract rows

table_data = []

rows = table.find("tbody").find_all("tr")

for row in rows:
    cols = row.find_all("td")
    row_data = [col.get_text(" ", strip=True) for col in cols]
    table_data.append(row_data)

# Convert to DataFrame
df = pd.DataFrame(table_data, columns=headers)

print(df)

# Save to CSV
df.to_csv("parivesh_data.csv", index=False)

   S. No.            Proposal No.  \
0       1  IA/AP/IND1/424318/2023   
1       2  IA/AP/IND1/414217/2023   
2       3  IA/AP/IND1/441309/2023   
3       4  IA/AP/IND1/400545/2022   
4       5  IA/AP/IND1/459174/2024   
..    ...                     ...   
62     63  IA/AP/IND1/521405/2025   
63     64  IA/AP/IND1/472021/2024   
64     65  IA/AP/IND1/505916/2024   
65     66  IA/AP/IND1/462848/2024   
66     67  IA/AP/IND1/441763/2023   

                                    Clearance Details  \
0   Clearance Type: Application for Transfer of EC...   
1   Clearance Type: Application for ToR (Category ...   
2   Clearance Type: Application for EC (Category A...   
3   Clearance Type: Application for EC (Category A...   
4   Clearance Type: Application for ToR (Category ...   
..                                                ...   
62  Clearance Type: Application for ToR (Category ...   
63  Clearance Type: Application for No Increase in...   
64  Clearance Type: Application for ToR (C

Link clicked successfully


In [ ]:
# driver.window_handles
# driver.switch_to.window(driver.window_handles[1])

# # Locate and click the link
# proposal_link = wait.until(
#     EC.element_to_be_clickable(
#         (
#             By.XPATH,
#             "//a[contains(text(),'IA/AP/IND1/424318/2023')]"
#         )
#     )
# )

# proposal_link.click()

# print("Link clicked successfully")


'https://parivesh.nic.in/newupgrade/#/trackYourProposal/proposal-details?proposalId=IA%2FAP%2FIND1%2F424318%2F2023&proposal=4332200'

In [ ]:
# Quit the WebDriver
driver.quit()

In [ ]:
# imports

import requests
from IPython.display import Markdown, display
import ollama

In [ ]:
# A class to represent a Webpage

class Website:
    """
    A utility class to represent a Website that we have scraped
    """
    url: str
    title: str
    text: str

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)
    
pr = Website("https://parivesh.nic.in/")

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "The contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

# And now: call the Ollama function instead of OpenAI

# Constants

MODEL = "llama3.2"

def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']


# A function to display this nicely in the Jupyter output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

summarize("https://parivesh.nic.in/")
display_summary("https://anthropic.com")

In [ ]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

display(response.choices[0].message.content)
def summarize(url):
    website = Website(url)
    messages = messages_for(website)
    response = ollama.chat(model=MODEL, messages=messages)
    return response['message']['content']
chrome_options = webdriver.ChromeOptions()